In [1]:
# pip install face_recognition opencv-python pyserial requests

In [3]:
import cv2
import face_recognition
import numpy as np
import requests
import serial
import time

# === Serial Setup ===
SERIAL_PORT = '/dev/ttyUSB0'  # Change to your port
BAUD_RATE = 115200            # Match your device baudrate

try:
    ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
    print(f"[INFO] Serial port {SERIAL_PORT} opened.")
except Exception as e:
    ser = None
    print(f"[WARNING] Could not open serial port {SERIAL_PORT}: {e}")

# === Known Faces Setup ===
known_faces_urls = {
    "Elon Musk": "https://upload.wikimedia.org/wikipedia/commons/4/49/Elon_Musk_2015.jpg",
    "Donald Trump": "https://upload.wikimedia.org/wikipedia/commons/5/56/Donald_Trump_official_portrait.jpg",
    # Add more known persons if you want
}

known_face_encodings = []
known_face_names = []

print("[INFO] Downloading and encoding known faces (in memory)...")

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/114.0.0.0 Safari/537.36"
}

for name, url in known_faces_urls.items():
    try:
        print(f"Downloading {name} image...")
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        img_array = np.asarray(bytearray(response.content), dtype=np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
        if img is None:
            raise ValueError("OpenCV failed to decode image")

        rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        encodings = face_recognition.face_encodings(rgb_img)
        if len(encodings) == 0:
            print(f"[WARNING] No face found in {name} image.")
            continue

        known_face_encodings.append(encodings[0])
        known_face_names.append(name)
        print(f"Encoded {name}")

    except Exception as e:
        print(f"[ERROR] Could not download or process {name}: {e}")

if len(known_face_encodings) == 0:
    print("[ERROR] No known face encodings available. Exiting.")
    exit(1)

print(f"[INFO] Total known faces encoded: {len(known_face_encodings)}")

# === Serial Commands (hex strings) ===
COMMANDS = {
    "Elon Musk": '3A01000201',  # Serial command #2 for Elon Musk
    "Donald Trump": '3A01010200', # Serial command #1 for Bill Gates
    "Unknown":   '3A01000200',  # Serial command for unknown face
}

# === Webcam Setup ===
print("[INFO] Starting webcam face recognition. Press 'q' to quit.")
video_capture = cv2.VideoCapture(0)
if not video_capture.isOpened():
    print("[ERROR] Cannot open webcam")
    exit(1)

process_this_frame = True

while True:
    ret, frame = video_capture.read()
    if not ret:
        print("[ERROR] Failed to capture frame from webcam")
        break

    # Resize frame for faster processing
    small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
    rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

    if process_this_frame:
        face_locations = face_recognition.face_locations(rgb_small_frame)
        face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)

        face_names = []
        for face_encoding in face_encodings:
            matches = face_recognition.compare_faces(known_face_encodings, face_encoding)
            name = "Unknown"

            face_distances = face_recognition.face_distance(known_face_encodings, face_encoding)
            if len(face_distances) > 0:
                best_match_index = np.argmin(face_distances)
                if matches[best_match_index]:
                    name = known_face_names[best_match_index]

            face_names.append(name)

            # Serial communication based on recognition
            if ser is not None:
                command = COMMANDS.get(name, COMMANDS["Unknown"])
                ser.write(bytes.fromhex(command))
                ser.flush()
                print(f"[SERIAL] Sent command for: {name}")
                time.sleep(0.2)  # Short delay for device to process

    process_this_frame = not process_this_frame

    # Display results
    for (top, right, bottom, left), name in zip(face_locations, face_names):
        top *= 4
        right *= 4
        bottom *= 4
        left *= 4

        cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)
        cv2.rectangle(frame, (left, bottom - 35), (right, bottom), (0, 255, 0), cv2.FILLED)
        cv2.putText(frame, name, (left + 6, bottom - 6),
                    cv2.FONT_HERSHEY_DUPLEX, 1.0, (0, 0, 0), 1)

    cv2.imshow("Face Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Cleanup
video_capture.release()
cv2.destroyAllWindows()

if ser is not None:
    ser.close()
    print("[INFO] Serial port closed.")


[INFO] Serial port /dev/ttyUSB0 opened.
[INFO] Downloading and encoding known faces (in memory)...
Encoded Elon Musk
Encoded Donald Trump
[INFO] Total known faces encoded: 2
[INFO] Starting webcam face recognition. Press 'q' to quit.
[SERIAL] Sent command for: Unknown
[SERIAL] Sent command for: Unknown
[SERIAL] Sent command for: Unknown
[SERIAL] Sent command for: Unknown
[SERIAL] Sent command for: Unknown
[SERIAL] Sent command for: Unknown
[SERIAL] Sent command for: Unknown
[SERIAL] Sent command for: Unknown
[SERIAL] Sent command for: Unknown
[SERIAL] Sent command for: Unknown
[SERIAL] Sent command for: Unknown
[SERIAL] Sent command for: Unknown
[SERIAL] Sent command for: Unknown
[SERIAL] Sent command for: Unknown
[SERIAL] Sent command for: Unknown
[SERIAL] Sent command for: Unknown
[SERIAL] Sent command for: Unknown
[SERIAL] Sent command for: Unknown
[SERIAL] Sent command for: Unknown
[SERIAL] Sent command for: Unknown
[SERIAL] Sent command for: Unknown
[SERIAL] Sent command for: Unkn